# Train News-enhanced Model

This notebook trains the same main `StockTransformer` architecture as the base model, but with FinBERT daily sentiment features added to the input.

The goal is a fair A/B setup:

- `01_train_model.ipynb` trains `best_model_base.pt` with technical features only.
- This notebook trains `best_model_news.pt` with technical features + news sentiment.
- `06_train_with_sentiment.ipynb` loads both checkpoints and compares them on the same test split.

The only intended modeling difference is `config.data.use_news = True`.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import os
os.chdir(project_root)

import torch
from torch.utils.data import DataLoader

from src.data.pipeline import get_datasets
from src.evaluation.visualizations import plot_training_curves
from src.models.transformer_model import StockTransformer
from src.training.trainer import Trainer
from src.utils.config import load_config, PROJECT_ROOT

In [ ]:
config = load_config()

# The only difference versus the base model training notebook.
config.data.use_news = True
config.paths.checkpoint_file = "best_model_news.pt"

print("Loading data with technical + FinBERT sentiment features...")
train_dataset, val_dataset, test_dataset, feature_columns = get_datasets(config)

news_cols = [col for col in feature_columns if col.startswith("news_")]
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")
print(f"Feature dims:  {len(feature_columns)}")
print(f"News features: {news_cols}")

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=config.training.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.training.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

model = StockTransformer(
    input_dim=len(feature_columns),
    d_model=config.model.d_model,
    n_heads=config.model.n_heads,
    n_layers=config.model.n_layers,
    d_ff=config.model.d_ff,
    dropout=config.model.dropout,
    activation=config.model.activation,
    prediction_horizon=config.data.prediction_horizon,
)

trainer = Trainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
history = trainer.train()

print(f"\nTraining finished")
print(f"Best validation loss: {history['best_val_loss']:.6f}")

In [ ]:
checkpoint_path = PROJECT_ROOT / config.paths.models_dir / "best_model_news.pt"

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Expected checkpoint was not created: {checkpoint_path}")

ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
print(f"Saved checkpoint: {checkpoint_path}")
print(f"Validation loss:  {ckpt.get('score', history['best_val_loss']):.6f}")
print(f"Model keys:       {len(ckpt['model_state_dict'])}")

results_dir = PROJECT_ROOT / config.paths.results_dir
results_dir.mkdir(parents=True, exist_ok=True)
plot_training_curves(
    history["train_losses"],
    history["val_losses"],
    save_path=results_dir / "training_curves_news.png",
)